In [22]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [23]:
from datasets import load_dataset

# Stream one English sample — no full download
ds = load_dataset(
    "amphion/Emilia-Dataset",
    split="train",
    streaming=True,
)
sample = next(iter(ds))

print("Keys:", list(sample.keys()))
print("Text:", sample.get("text", sample.get("json", {}).get("text", "—")))

audio = sample["mp3"]  # dict with 'array' and 'sampling_rate'
print(
    f"Sample rate: {audio['sampling_rate']} Hz, length: {len(audio['array'])} samples"
)

Resolving data files:   0%|          | 0/4343 [00:00<?, ?it/s]

Keys: ['json', 'mp3', '__key__', '__url__']
Text:  So. Chloe hat gesagt, ich soll noch unten gehen. Was ich natürlich auch machen werde.
Sample rate: 24000 Hz, length: 193968 samples


In [24]:
import torch

# transform audio to tensor and add dimension
audio_tensor = torch.tensor(audio["array"]).unsqueeze(0).float()
sr = audio["sampling_rate"]
print(f"Audio tensor: {audio_tensor.shape}, sr={sr}")

Audio tensor: torch.Size([1, 193968]), sr=24000


In [25]:
from TTS.api import TTS

# bypass coqui aggrement
os.environ["COQUI_TOS_AGREED"] = "1"
# Downloads and caches to ~/.local/share/tts/ on first run
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
model = tts.synthesizer.tts_model
print("Model loaded:", type(model).__name__)

Model loaded: Xtts


In [26]:
gpt_cond_latent = model.get_gpt_cond_latents(
    audio_tensor, sr, length=model.config.gpt_cond_len
)
speaker_embedding = model.get_speaker_embedding(audio_tensor, sr)

print("gpt_cond_latent shape:", gpt_cond_latent.shape)
print("speaker_embedding shape:", speaker_embedding.shape)

gpt_cond_latent shape: torch.Size([1, 32, 1024])
speaker_embedding shape: torch.Size([1, 512, 1])


---

# Temporary Example extracting "directions" for pitch etc.

In [27]:
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import numpy as np
import torch
import librosa


def load_or_generate_directions(
        *,
        create_new: bool,
        ds: Optional[Iterable[dict]] = None,
        model: Optional[Any] = None,
        max_scan: int = 1000,
        pct: float = 0.15,
        save_dir: str | Path = "directions",
        load_path: str | Path = None,
        save: bool = True,
        filename_prefix: str = "xtts_directions",
        device: Optional[str] = None,
        include_speaker: bool = True,  # <--- NEW: also learn directions in speaker-embedding space
) -> Dict[str, Any]:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if not create_new:
        load_path = Path(load_path)
        payload = torch.load(load_path, map_location=device)
        print(f"Loaded directions from: {load_path}")
        return payload

    # --- Collect representations + stats ---
    all_latents: list[torch.Tensor] = []
    all_spk_embs: list[torch.Tensor] = []  # <--- NEW
    stats: list[dict] = []

    print(f"Scanning {max_scan} samples... (pct={pct:.3f}, device={device}, include_speaker={include_speaker})")

    processed = 0
    for i, sample in enumerate(ds):
        if i >= max_scan:
            break

        audio_array = np.asarray(sample["mp3"]["array"])
        sr = int(sample["mp3"]["sampling_rate"])
        text = sample.get("text", "") or sample.get("json", {}).get("text", "") or ""

        duration = len(audio_array) / sr if sr > 0 else 0.0
        cps = (len(text) / duration) if duration > 0 else 0.0

        f0 = librosa.yin(
            audio_array,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
        )
        mean_pitch = float(np.mean(f0[f0 > 0])) if np.any(f0 > 0) else 0.0

        audio_tensor = torch.tensor(audio_array).unsqueeze(0).float().to(device)
        with torch.no_grad():
            latent = model.get_gpt_cond_latents(
                audio_tensor,
                sr,
                length=model.config.gpt_cond_len,
            )
            all_latents.append(latent)

            if include_speaker:
                spk = model.get_speaker_embedding(audio_tensor, sr)
                all_spk_embs.append(spk)

        stats.append({"cps": float(cps), "pitch": float(mean_pitch)})
        processed += 1

        if (processed % 25 == 0):
            print(f"  Processed {processed}/{max_scan}...")

    if processed == 0:
        raise RuntimeError("Processed 0 samples; dataset iterator may be empty or failing")

    def _direction_for(feature_name: str, reps: list[torch.Tensor]) -> torch.Tensor:
        values = np.array([s[feature_name] for s in stats], dtype=np.float64)
        indices = np.argsort(values)

        num = max(1, int(processed * pct))
        low_idx = indices[:num]
        high_idx = indices[-num:]

        print(f"Calculating {feature_name}: averaging {len(low_idx)} low vs {len(high_idx)} high samples")

        mean_low = torch.stack([reps[int(j)] for j in low_idx]).mean(dim=0)
        mean_high = torch.stack([reps[int(j)] for j in high_idx]).mean(dim=0)
        return mean_high - mean_low

    # GPT-latent directions (existing behavior)
    speed_direction = _direction_for("cps", all_latents)
    pitch_direction = _direction_for("pitch", all_latents)

    payload: Dict[str, Any] = {
        "speed_direction": speed_direction.detach(),
        "pitch_direction": pitch_direction.detach(),
        # Helpful to keep around if you want to compute anchor points later without re-scanning:
        "stats": stats,
        "all_latents": [t.detach().cpu() for t in all_latents],
    }

    # Speaker-embedding directions (NEW)
    if include_speaker:
        spk_speed_direction = _direction_for("cps", all_spk_embs)
        spk_pitch_direction = _direction_for("pitch", all_spk_embs)
        payload.update(
            {
                "spk_speed_direction": spk_speed_direction.detach(),
                "spk_pitch_direction": spk_pitch_direction.detach(),
                "all_spk_embs": [t.detach().cpu() for t in all_spk_embs],
            }
        )

    created_at = datetime.now().strftime("%Y%m%d_%H%M%S")
    pct_label = f"{pct * 100:.1f}".replace(".", "p")  # e.g., 15.0 -> "15p0"
    filename = f"{filename_prefix}_N{processed}_pct{pct_label}_{created_at}.pt"

    if save:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / filename
        torch.save(payload, save_path)
        print(f"Saved directions to: {save_path}")

    return payload


In [28]:
CREATE_NEW = False
SAVE = False

directions = load_or_generate_directions(
    create_new=CREATE_NEW,
    save=SAVE,
    ds=ds,
    model=model,
    max_scan=1000,
    pct=0.15,
    save_dir="directions",
    load_path=r"directions\xtts_directions_N1000_pct15p0_20260226_111738.pt",
    include_speaker=True,
)

speed_direction = directions["speed_direction"]
pitch_direction = directions["pitch_direction"]

# speaker embedding directions
spk_speed_direction = directions.get("spk_speed_direction", None)
spk_pitch_direction = directions.get("spk_pitch_direction", None)

Loaded directions from: directions\xtts_directions_N1000_pct15p0_20260226_111738.pt


In [29]:
print(torch.norm(speed_direction).item())
print(torch.norm(pitch_direction).item())
print(torch.norm(spk_speed_direction).item())
print(torch.norm(spk_pitch_direction).item())

51.77907943725586
72.02571105957031
0.25594672560691833
0.4154522716999054


In [30]:
from dataclasses import dataclass

@dataclass
class DirectionsBundle:
    # GPT (Latent) Directions/Targets
    speed_direction: torch.Tensor | None = None
    pitch_direction: torch.Tensor | None = None
    speed_low: torch.Tensor | None = None
    speed_high: torch.Tensor | None = None
    pitch_low: torch.Tensor | None = None
    pitch_high: torch.Tensor | None = None

    # Speaker Embedding Directions/Targets
    spk_speed_direction: torch.Tensor | None = None
    spk_pitch_direction: torch.Tensor | None = None
    spk_speed_low: torch.Tensor | None = None
    spk_speed_high: torch.Tensor | None = None
    spk_pitch_low: torch.Tensor | None = None
    spk_pitch_high: torch.Tensor | None = None


def slerp(val, low, high):
    """
    Spherical linear interpolation between two tensors (low and high)
    based on 'val' (0.0 to 1.0).
    """
    low_norm = low / torch.norm(low)
    high_norm = high / torch.norm(high)

    dot = torch.sum(low_norm * high_norm)
    dot = torch.clamp(dot, -1.0, 1.0)
    omega = torch.acos(dot)
    so = torch.sin(omega)

    if so < 1e-6:
        return (1.0 - val) * low + val * high

    return (torch.sin((1.0 - val) * omega) / so) * low + (torch.sin(val * omega) / so) * high


def get_anchor_points(feature_name: str, reps: list[torch.Tensor], stats_list: list[dict], *, pct: float = 0.15):
    values = [s[feature_name] for s in stats_list]
    indices = np.argsort(values)
    num = max(1, int(len(indices) * pct))

    low_idx = indices[:num]
    high_idx = indices[-num:]

    print(f"Anchoring {feature_name}: using {len(low_idx)} samples for Low and High points")

    mean_low = torch.stack([reps[int(i)] for i in low_idx]).mean(dim=0)
    mean_high = torch.stack([reps[int(i)] for i in high_idx]).mean(dim=0)
    return mean_low, mean_high


stats = directions.get("stats", None)
all_latents = directions.get("all_latents", None)
all_spk_embs = directions.get("all_spk_embs", None)

speed_low, speed_high = get_anchor_points("cps", all_latents, stats, pct=0.15)
pitch_low, pitch_high = get_anchor_points("pitch", all_latents, stats, pct=0.15)
spk_speed_low, spk_speed_high = get_anchor_points("cps", all_spk_embs, stats, pct=0.15)
spk_pitch_low, spk_pitch_high = get_anchor_points("pitch", all_spk_embs, stats, pct=0.15)

directions_bundle = DirectionsBundle(
    speed_direction=speed_direction,
    pitch_direction=pitch_direction,
    speed_low=speed_low,
    speed_high=speed_high,
    pitch_low=pitch_low,
    pitch_high=pitch_high,
    spk_speed_direction=spk_speed_direction,
    spk_pitch_direction=spk_pitch_direction,
    spk_speed_low=spk_speed_low,
    spk_speed_high=spk_speed_high,
    spk_pitch_low=spk_pitch_low,
    spk_pitch_high=spk_pitch_high,
)


Anchoring cps: using 150 samples for Low and High points
Anchoring pitch: using 150 samples for Low and High points
Anchoring cps: using 150 samples for Low and High points
Anchoring pitch: using 150 samples for Low and High points


In [31]:
from IPython.display import Audio

def compute_modified_embeddings(
    *,
    base_gpt_cond_latent: torch.Tensor,
    base_speaker_embedding: torch.Tensor,
    directions_bundle: DirectionsBundle,
    mode: str,
    method: str,
    speed: float,
    pitch: float,
    slider_abs_max: float = 1.0,
    normalize_speaker_embedding: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:

    s = float(speed)
    p = float(pitch)
    latent = base_gpt_cond_latent
    spk = base_speaker_embedding

    if method == "linear":
        if mode == "gpt":
            latent = base_gpt_cond_latent + (s * directions_bundle.speed_direction) + (p * directions_bundle.pitch_direction)
        elif mode == "spk":
            spk = base_speaker_embedding + (s * directions_bundle.spk_speed_direction) + (p * directions_bundle.spk_pitch_direction)
            if normalize_speaker_embedding:
                spk = torch.nn.functional.normalize(spk, p=2, dim=1)
    elif method == "spherical":
        if mode == "gpt":
            if s != 0:
                target = directions_bundle.speed_high if s > 0 else directions_bundle.speed_low
                latent = slerp(min(abs(s) / slider_abs_max, 1.0), latent, target)
            if p != 0:
                target = directions_bundle.pitch_high if p > 0 else directions_bundle.pitch_low
                latent = slerp(min(abs(p) / slider_abs_max, 1.0), latent, target)

        elif mode == "spk":
            if s != 0:
                target = directions_bundle.spk_speed_high if s > 0 else directions_bundle.spk_speed_low
                spk = slerp(min(abs(s) / slider_abs_max, 1.0), spk, target)
            if p != 0:
                target = directions_bundle.spk_pitch_high if p > 0 else directions_bundle.spk_pitch_low
                spk = slerp(min(abs(p) / slider_abs_max, 1.0), spk, target)
            if normalize_speaker_embedding:
                spk = torch.nn.functional.normalize(spk, p=2, dim=1)

    return latent, spk


def synthesize(
    *,
    model: Any,
    base_gpt_cond_latent: torch.Tensor,
    base_speaker_embedding: torch.Tensor,
    directions_bundle: DirectionsBundle,
    mode: str,
    method: str,
    speed: float,
    pitch: float,
    slider_abs_max: float = 1.0,
    autoplay: bool,
) -> dict:

    modified_latent, modified_spk = compute_modified_embeddings(
        base_gpt_cond_latent=base_gpt_cond_latent,
        base_speaker_embedding=base_speaker_embedding,
        directions_bundle=directions_bundle,
        mode=mode,
        method=method,
        speed=speed,
        pitch=pitch,
        slider_abs_max=slider_abs_max,
    )

    with torch.no_grad():
        out = model.inference(
            text="Hello! This is how I sound after you modified my embeddings.",
            language="en",
            gpt_cond_latent=modified_latent,
            speaker_embedding=modified_spk,
            temperature=model.config.temperature,
            length_penalty=model.config.length_penalty,
            repetition_penalty=model.config.repetition_penalty,
            top_k=model.config.top_k,
            top_p=model.config.top_p,
        )

    # Optional convenience: play audio if we're in a notebook context
    if "wav" in out:
        display(Audio(out["wav"], rate=24000, autoplay=autoplay))
    return out

In [32]:
# UI Elements
from ipywidgets import widgets
from IPython.display import Audio, display, clear_output
s_slider = widgets.FloatSlider(value=0, min=-10, max=10, step=0.2, description="Speed 🏃")
p_slider = widgets.FloatSlider(value=0, min=-10, max=10, step=0.2, description="Pitch 🎤")
apply_to = widgets.ToggleButtons(options=[("Conditional Latent", "gpt"), ("Speaker Embedding", "spk")], value="gpt", description="Apply to:",)
linear_btn = widgets.Button(description="Generate Linear", button_style="info", icon="play")
spherical_btn = widgets.Button(description="Generate Spherical", button_style="warning", icon="play")
ui_out = widgets.Output()

def synthesize_from_ui(*, method: str) -> None:
    with ui_out:
        clear_output()
        mode = apply_to.value  # "gpt" or "spk"
        s = float(s_slider.value)
        p = float(p_slider.value)

        print(f"Rendering ({method.title()}, {mode.upper()})  S:{s}  P:{p} ...")

        synthesize(
            model=model,
            base_gpt_cond_latent=gpt_cond_latent,
            base_speaker_embedding=speaker_embedding,
            directions_bundle=directions_bundle,
            mode=apply_to.value,
            method=method,
            speed=s_slider.value,
            pitch=p_slider.value,
        )

# Buttons
def synthesize_linear(_b):
    synthesize_from_ui(method="linear")

def synthesize_spherical(_b):
    synthesize_from_ui(method="spherical")

linear_btn.on_click(synthesize_linear)
spherical_btn.on_click(synthesize_spherical)

display(
    widgets.VBox(
        [
            apply_to,
            s_slider,
            p_slider,
            widgets.HBox([linear_btn, spherical_btn]),
            ui_out,
        ]
    )
)

In [33]:
from pathlib import Path
from typing import Any, Literal, Sequence
import numpy as np
import torch
from scipy.io import wavfile

Control = Literal["speed", "pitch", "none"]
Method = Literal["linear", "spherical"]

def create_many_samples(
    *,
    model: Any,
    base_gpt_cond_latent: torch.Tensor,
    base_speaker_embedding: torch.Tensor,
    directions_bundle=directions_bundle,
    mode: Literal["gpt", "spk"],
    method: Method,
    n_per_setting: int,
    values: Sequence[float],
    control: Control,
    seed: int | None = None,           # set for reproducibility
    save_dir: str | Path | None = None,
    slider_abs_max: float = 1.0,
):
    """
    Generate many audio samples for a sweep of values.

    Inputs
      - n_per_setting: how many samples per value
      - values: list of values to use, e.g. [-1, -0.5, 0, 0.5, 2]
      - control: "speed" or "pitch" or "none"
      - method: "linear" or "spherical"
    """

    if control == "none":
        sweep_values = [0.0]
    else:
        sweep_values = [float(v) for v in values]

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

    base_seed = seed
    for rep_idx in range(n_per_setting):
        for setting_idx, v in enumerate(sweep_values):
            if base_seed is not None:
                cur_seed = int(base_seed + setting_idx * 10_000 + rep_idx)
                torch.manual_seed(cur_seed)
                np.random.seed(cur_seed % (2**32 - 1))

            speed = 0.0
            pitch = 0.0
            if control == "speed":
                speed = float(v)
            elif control == "pitch":
                pitch = float(v)

            if save_dir is not None:
                safe_control = control
                safe_method = method
                fname = (
                    f"{mode}"
                    f"_{safe_method}"
                    f"_{safe_control}"
                    f"_{float(v):+.3f}"
                    f"_{rep_idx:02d}"
                    f".wav"
                )
                path = save_dir / fname

                # Only synthesize and save a sample if it doesn't exist yet
                if not path.exists():
                    out = synthesize(
                        model=model,
                        base_gpt_cond_latent=base_gpt_cond_latent,
                        base_speaker_embedding=base_speaker_embedding,
                        directions_bundle=directions_bundle,
                        mode=mode,
                        method=method,
                        speed=speed,
                        pitch=pitch,
                        slider_abs_max=slider_abs_max,
                        autoplay=False,
                    )
                    wav = np.asarray(out["wav"], dtype=np.float32)
                    wavfile.write(path, 24000, wav)

    return

In [ ]:
'''
Values

speed, linear: Plus Seems to speed up recording
+5 sounds like male
+10 sounds like male
+100 too much, garbage
-100 too much, garbage

pitch, linear:
Seems to hear most difference in range 2, 3, 5, 8
More is not helpful
+50 too much, garbage
-50 too much, garbage

'''

In [36]:
create_many_samples(
    model=model,
    base_gpt_cond_latent=gpt_cond_latent,
    base_speaker_embedding=speaker_embedding,
    directions_bundle=directions_bundle,
    mode="gpt",                          # "gpt" or "spk"
    method="linear",                  # "linear" or "spherical"
    n_per_setting=1,
    values=[-50, -10, -8, -5, -3, -2, -1, -0.5, -0.1, -0.01, 0, 0.01, 0.1, 0.5, 1, 2, 3, 5, 8, 10, 50],
    control="pitch",                     # "speed" or "pitch"
    seed=123,
    save_dir="renders",
)
